## 실험의 의의 
- AI 한번 돌리는데 너무 많은 연산과 컴퓨팅 파워가 소모된다...돈없는 사람은 AI 도 못쓰나?
- 단순히 점수 가장 높은 모델이 아니라, **적은 메모리와 연산**으로 어느 정도까지 쓸 만한 성능을 낼 수 있는지를 확인 하자
- 이를 위해 여러 ML 모델과 Dense 딥러닝 모델을, 같은 TF‑IDF 표현 위에서, 단어장(vocabulary) 크기를 바꿔가며 비교

## 회고
- 전체적으로, 딥러닝이  좋을 것이라는 편견과 달리 전통 ML(특히 선형 모델)이 적은 자원으로 충분히 강력한 대안이 될 수 있을 것으로 보임
- **LinearSVC** 는 모든 단어장 크기에서 가장 안정적으로 높은 F1-score 기록하면서도  학습·추론 시간이 매우 짧아, **가성비 베이스라인 모델**로 고려 가능
- 단어장을 2000개로 줄여도 LinearSVC와 XGBoost의 성능은 거의 유지, DenseNN 역시 약간의 성능 저하만 보임
- XGBoost와 DenseNN은 LinearSVC와 비슷한 수준의 성능이긴 하지만, 학습 시간과 모델 크기 측면에서 더 무거워 로컬/저성능 환경을 고려하면 좋은 선택이 아님



### 단어장 개수별 모델 성능 + 효율 (Accuracy / F1 / TrainTime_s / ModelSize_KB)

| Vocabulary | Model              | Accuracy | F1 | TrainTime_s | ModelSize_KB |
|-----------:|--------------------|---------:|---------:|------------:|-------------:|
| 2000       | LogisticRegression | 0.8019   | 0.4813   | 3.32        | 720.29       |
|            | LinearSVC          | 0.8264   | 0.6605   | 1.89        | 720.16       |
|            | RandomForest       | 0.7743   | 0.4763   | 6.52        | 323617.07    |
|            | XGBoost            | 0.8108   | 0.6227   | 154.93      | 10825.12     |
|            | NaiveBayes         | 0.6963   | 0.1728   | 0.01        | 1439.30      |
|            | LightGBM           | 0.3468   | 0.0119   | 24.16       | 4022.35      |
|            | DecisionTree       | 0.6955   | 0.4418   | 3.51        | 1265.99      |
|            | DenseNN            | 0.7907   | 0.5865   | 19.03       | 12876.45     |
| 5000       | LogisticRegression | 0.7979   | 0.4770   | 3.93        | 1798.41      |
|            | LinearSVC          | 0.8299   | 0.6840   | 2.22        | 1798.28      |
|            | RandomForest       | 0.7654   | 0.4434   | 9.52        | 356979.79    |
|            | XGBoost            | 0.8126   | 0.6513   | 200.94      | 10743.44     |
|            | NaiveBayes         | 0.6745   | 0.1107   | 0.01        | 3595.55      |
|            | LightGBM           | 0.1861   | 0.0085   | 29.71       | 4447.22      |
|            | DecisionTree       | 0.6883   | 0.4432   | 4.79        | 1250.81      |
|            | DenseNN            | 0.8037   | 0.6164   | 16.64       | 30876.45     |
| 10000      | LogisticRegression | 0.7956   | 0.4721   | 5.02        | 3476.69      |
|            | LinearSVC          | 0.8299   | 0.6808   | 0.90        | 3476.57      |
|            | RandomForest       | 0.7573   | 0.4476   | 13.75       | 386640.98    |
|            | XGBoost            | 0.8108   | 0.6294   | 286.34      | 10744.73     |
|            | NaiveBayes         | 0.6567   | 0.0967   | 0.02        | 6952.12      |
|            | LightGBM           | 0.3998   | 0.0244   | 31.27       | 4875.91      |
|            | DecisionTree       | 0.6968   | 0.4603   | 4.50        | 1224.65      |
|            | DenseNN            | 0.8090   | 0.6249   | 18.14       | 58896.45     |
| All        | LogisticRegression | 0.7956   | 0.4721   | 5.65        | 3476.69      |
|            | LinearSVC          | 0.8299   | 0.6808   | 0.71        | 3476.57      |
|            | RandomForest       | 0.7573   | 0.4476   | 13.08       | 386640.98    |
|            | XGBoost            | 0.8108   | 0.6294   | 295.45      | 10744.73     |
|            | NaiveBayes         | 0.6567   | 0.0967   | 0.02        | 6952.12      |
|            | LightGBM           | 0.3998   | 0.0244   | 31.95       | 4875.88      |
|            | DecisionTree       | 0.6968   | 0.4603   | 4.52        | 1224.65      |
|            | DenseNN            | 0.8090   | 0.6173   | 16.05       | 58896.45     |


# 실험 설계



### 1. 실험 목표

1. 단어장 크기(vocabulary size)에 따른 성능·효율 변화 관찰 
   - `max_features = 10000, 5000, None(All)`로 단어장 크기를 바꿨을 때  
   - Accuracy와 F1‑score가 어떻게 달라지는지 확인  
   - 성능을 거의 잃지 않고 줄일 수 있는 최소 단어장 크기에 대한 감을 잡기

2. 같은 TF‑IDF 표현에서 여러 ML 모델의 특성 비교
   - Logistic Regression, SVM(LinearSVC), RandomForest, XGBoost, Naive Bayes, LightGBM, DecisionTree, Dense NN  
   - 각 모델의 Accuracy, F1‑score를 비교하고  
   - 어떤 모델이 가볍고 빠르면서도 성능이 괜찮은지 파악

3. 딥러닝(Dense) vs 전통 머신러닝의 차이 이해
   - 같은 TF‑IDF 입력을 사용했을 때, Dense 모델과 전통 ML(Logistic Regression, SVM 등)의 성능과 효율을 비교  

4. 저성능, 개인용 컴퓨터 환경을 염두에 둔 경량 모델 감각 익히기
   - 제한된 메모리와 연산 자원(로컬 LLM, 온디바이스 AI 등)을 고려했을 때  
   - TF‑IDF + 전통 ML 조합이 얼마나 강력한 베이스라인인지 체감  
   - 성능 vs 모델 크기 vs 속도 사이의 트레이드오프를 확인

---

### 2. 실험 설계

- 데이터셋 : Keras Reuters 뉴스 데이터 (다중 분류, 46 클래스)  
- 입력 표현:  
  - 인덱스를 단어로 디코딩한 후  `CountVectorizer(max_features=...)` + `TfidfTransformer()`로 TF‑IDF 벡터화  
- Vocabulary Size:  
  - 2,000/ 10,000 / 5,000 / None(All words)  
- 비교할 모델 (총 8개):  
  - Logistic Regression  
  - SVM (LinearSVC)  
  - RandomForest  
  - XGBoost  
  - Naive Bayes (MultinomialNB)  
  - Dense NN (Keras)  
  - LightGBM  
  - DecisionTree  
- 평가지표:  
  - Accuracy  
  - F1‑score (다중 분류 → macro 또는 weighted, 실험 시 명시)
  - Train Time (s): `fit()`에 걸린 학습 시간 (초 단위 측정)  
  - Test Time (s):  전체 테스트 데이터에 대한 `predict()` 시간  
  - Model Size (rough):  
    - 전통 ML: 학습된 모델 객체를 디스크에 저장했을 때 용량 (예: `joblib.dump` 후 파일 크기)  
    - Dense NN: `model.count_params()`로 파라미터 수, 또는 저장된 `.h5` 파일 크기  

---

### 3. 기대하는 인사이트

- 단어장 크기를 줄여도 성능이 거의 유지되는 구간을 찾는다.  
- TF‑IDF 기반 텍스트 분류에서, 어떤 모델들이 “성능/무게/속도” 균형이 좋은지 정리한다.  
- Dense 딥러닝 모델이 전통 ML 대비 가지는 장단점을, 실제 수치와 함께 설명할 수 있게 된다.  
- 향후 로컬 LLM이나 온디바이스 NLP를 설계할 때, “최소한 이 정도 베이스라인(LogReg/SVM + TF‑IDF + 제한된 vocab)은 깔고 간다”라는 감각을 갖게 된다.

# 01_환경 설정 

In [1]:
import sys
print(sys.executable)

/usr/bin/python3


In [2]:
import time
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.datasets import reuters


I0000 00:00:1773292146.189238  128526 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# 02_데이터 로드 및 디코딩
- index -> text
- DTM , TF-idf 학습데이터 준비

In [3]:
num_words_for_load = 10000
(x_train_idx, y_train), (x_test_idx, y_test) = reuters.load_data(
    num_words=num_words_for_load, test_split=0.2)

word_index = reuters.get_word_index(path="reuters_word_index.json")

index_to_word = {index + 3: word for word, index in word_index.items()}
for index, token in enumerate(("<pad>", "<sos>", "<unk>")):
    index_to_word[index] = token

def decode_seq(seqs, index_to_word):
    decoded = []
    for s in seqs:
        decoded.append(" ".join(index_to_word.get(i, "<unk>") for i in s))
    return decoded

x_train_text = decode_seq(x_train_idx, index_to_word)
x_test_text = decode_seq(x_test_idx, index_to_word)

print("Train samples:", len(x_train_text))
print("Test samples:", len(x_test_text))

Train samples: 8982
Test samples: 2246


## 02_실험설정

In [ ]:
# 결과 모음 -> 한번 만들고 건들지 말것 
results = []

In [5]:
# 전통 ML / 트리 / 앙상블 모델 (Dense는 따로 처리)
def get_ml_models():
    N_JOBS = 6 # or 6 정도, 로컬 PC CPU 점유 상태 보면서 조절

    return {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            multi_class="multinomial",
            solver="lbfgs",
            n_jobs=N_JOBS,   # -1 -> 4
        ),
        "LinearSVC": LinearSVC(),  # n_jobs 옵션 없음

        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=N_JOBS,   # -1 -> 4
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=N_JOBS,   # -1 -> 4
            # GPU 쓰고 싶으면:
            # device="cuda",
        ),
        "NaiveBayes": MultinomialNB(),
        "LightGBM": LGBMClassifier(
            n_estimators=200,
            learning_rate=0.1,
            num_leaves=31,
            random_state=42,
            n_jobs=N_JOBS,   # 기본값 CPU 풀 → 4로 제한
        ),
        "DecisionTree": DecisionTreeClassifier(
            max_depth=None,
            random_state=42,
        ),
    }

# 03_실험 

## 03_Vocabulary size 실험

In [16]:
# vocab = 2000          # 첫 번째 
# vocab = 5000        # 두 번째 
# vocab = 10000       # 세 번째
vocab = None        # 네 번째

vocab_label = "All" if vocab is None else str(vocab)

print("\n", "="*60)
print(f"Vocab Size: {vocab}")
print("="*60)

# CountVectorizer 설정
if vocab is None:
    dtmvector = CountVectorizer()  # All words
else:
    dtmvector = CountVectorizer(max_features=vocab)

tfidf_transformer = TfidfTransformer()

# TF-IDF 만들기
t0 = time.time()
x_train_dtm = dtmvector.fit_transform(x_train_text)
x_train_tfidf = tfidf_transformer.fit_transform(x_train_dtm)
tfidf_train_time = time.time() - t0

t0 = time.time()
x_test_dtm = dtmvector.transform(x_test_text)
x_test_tfidf = tfidf_transformer.transform(x_test_dtm)
tfidf_test_time = time.time() - t0

print(f"TF-IDF train shape: {x_train_tfidf.shape}")
print(f"TF-IDF test  shape: {x_test_tfidf.shape}")

n_features = x_train_tfidf.shape[1]


Vocab Size: None
TF-IDF train shape: (8982, 9670)
TF-IDF test  shape: (2246, 9670)


## 03-1. 전통 ML + 앙상블 모델들 

In [17]:
ml_models = get_ml_models()
for model_name, model in ml_models.items():
    print(f"\n[ML] Vocab={vocab_label}, Model={model_name}")

    t0 = time.time()
    model.fit(x_train_tfidf, y_train)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = model.predict(x_test_tfidf)
    test_time = time.time() - t0

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    model_path = f"models/{model_name}_vocab_{vocab_label}.joblib"
    joblib.dump(model, model_path)
    model_size_kb = os.path.getsize(model_path) / 1024

    results.append({
        "Vocabulary": vocab_label,
        "Model": model_name,
        "Accuracy": acc,
        "F1_macro": f1_macro,
        "TrainTime_s": train_time,
        "TestTime_s": test_time,
        "ModelSize_KB": model_size_kb,
        "NumFeatures": n_features,
        "TFIDF_TrainTime_s": tfidf_train_time,
        "TFIDF_TestTime_s": tfidf_test_time,
    })


[ML] Vocab=All, Model=LogisticRegression


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



[ML] Vocab=All, Model=LinearSVC

[ML] Vocab=All, Model=RandomForest

[ML] Vocab=All, Model=XGBoost

[ML] Vocab=All, Model=NaiveBayes

[ML] Vocab=All, Model=LightGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.050017 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 163467
[LightGBM] [Info] Number of data points in the train set: 8982, number of used features: 3966
[LightGBM] [Info] Start training from score -5.095645
[LightGBM] [Info] Start training from score -3.034552
[LightGBM] [Info] Start training from score -4.798913
[LightGBM] [Info] Start training from score -1.044967
[LightGBM] [Info] Start training from score -1.527906
[LightGBM] [Info] Start training from score -6.269765
[LightGBM] [Info] Start training from score -5.231777
[LightGBM] [Info] Start training from score -6.330389
[LightGBM] [Info] Start training from score -4.168504
[LightGBM] [Info] Start training from score -4.487857
[L

/usr/local/lib/python3.10/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[ML] Vocab=All, Model=DecisionTree


## 03-2. Dense NN (딥러닝) 모델 실험

In [18]:
print(f"\n[DL] Vocab={vocab_label}, Model=DenseNN")

input_dim = n_features
num_classes = np.max(y_train) + 1

inputs = Input(shape=(input_dim,))
x = Dense(512, activation="relu")(inputs)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation="softmax")(x)

dense_model = Model(inputs=inputs, outputs=outputs)
dense_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

x_train_dense = x_train_tfidf.toarray()
x_test_dense = x_test_tfidf.toarray()

t0 = time.time()
history = dense_model.fit(
    x_train_dense,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)
train_time = time.time() - t0

t0 = time.time()
y_proba = dense_model.predict(x_test_dense, verbose=0)
test_time = time.time() - t0

y_pred = np.argmax(y_proba, axis=1)
acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")

num_params = dense_model.count_params()
dense_path = f"models/DenseNN_vocab_{vocab_label}.h5"
dense_model.save(dense_path)
model_size_kb = os.path.getsize(dense_path) / 1024

results.append({
    "Vocabulary": vocab_label,
    "Model": "DenseNN",
    "Accuracy": acc,
    "F1_macro": f1_macro,
    "TrainTime_s": train_time,
    "TestTime_s": test_time,
    "ModelSize_KB": model_size_kb,
    "NumParams": num_params,
    "NumFeatures": n_features,
    "TFIDF_TrainTime_s": tfidf_train_time,
    "TFIDF_TestTime_s": tfidf_test_time,
})


[DL] Vocab=All, Model=DenseNN
Epoch 1/10


I0000 00:00:1773293463.685762  130444 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_42991__.15


214/225 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4920 - loss: 2.3215

I0000 00:00:1773293466.046808  130443 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_42991__.15


225/225 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.6120 - loss: 1.6899 - val_accuracy: 0.7507 - val_loss: 1.1104
Epoch 2/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8100 - loss: 0.8251 - val_accuracy: 0.8036 - val_loss: 0.8737
Epoch 3/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8852 - loss: 0.4846 - val_accuracy: 0.8158 - val_loss: 0.8430
Epoch 4/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9301 - loss: 0.2950 - val_accuracy: 0.8275 - val_loss: 0.8241
Epoch 5/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9496 - loss: 0.2133 - val_accuracy: 0.8175 - val_loss: 0.8624
Epoch 6/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9556 - loss: 0.1684 - val_accuracy: 0.8136 - val_loss: 0.8768
Epoch 7/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9588 - loss: 0.1409 - val_accuracy: 0.8158 - val_loss: 0.9267
Epoch 8/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9594 - loss: 0.1348 - val_accuracy: 0.8114 - val

# 04_결과 정리

In [19]:
# 04. 결과 정리 --------------------------------------------------------

results_df = pd.DataFrame(results)

# 과제에서 요구한 형태의 핵심 표 (Accuracy / F1만)
pivot_main = results_df.pivot_table(
    index=["Vocabulary", "Model"],
    values=["Accuracy", "F1_macro"],
)

print("\n=== 성능 요약 (Accuracy / F1_macro) ===")
print(pivot_main)

# 효율 지표까지 포함한 전체 결과
print("\n=== 전체 결과 (성능 + 시간 + 모델 크기) ===")
print(results_df)

# 필요하면 CSV로 저장
results_df.to_csv("experiment_results_reuters_tfidf_models.csv", index=False)


=== 성능 요약 (Accuracy / F1_macro) ===
                               Accuracy  F1_macro
Vocabulary Model                                 
10000      DecisionTree        0.696794  0.460283
           DenseNN             0.808994  0.624908
           LightGBM            0.399822  0.024397
           LinearSVC           0.829920  0.680843
           LogisticRegression  0.795637  0.472114
           NaiveBayes          0.656723  0.096728
           RandomForest        0.757346  0.447596
           XGBoost             0.810775  0.629394
2000       DecisionTree        0.695459  0.441812
           DenseNN             0.790739  0.586457
           LightGBM            0.346839  0.011860
           LinearSVC           0.826358  0.660507
           LogisticRegression  0.801870  0.481296
           NaiveBayes          0.696349  0.172815
           RandomForest        0.774265  0.476257
           XGBoost             0.810775  0.622687
5000       DecisionTree        0.688335  0.443249
           De